## Milvus Vector DB for RAG 

### Create a collection of legal docs

In [1]:
%pip install pymilvus


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
import os
from dotenv import load_dotenv
# from pymilvus import connections

# # If using Docker standalone Milvus
# connections.connect("default", host="127.0.0.1", port="19530")

from pymilvus import connections

load_dotenv(override=True, dotenv_path="../.env.local")

milvus_uri = os.getenv("MILVUS_URI")
milvus_token = os.getenv("MILVUS_API_KEY")


connections.connect(
    alias="default",
    uri=milvus_uri,
    token=milvus_token
)

print("Connected to Milvus on Zilliz Cloud")


/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_13525/3686056658.py:16: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(


Connected to Milvus on Zilliz Cloud


In [8]:
from pymilvus import db
from pymilvus import Collection, FieldSchema, CollectionSchema, DataType

# 1. Create a new database
# db.create_database("rag_db")

# 2. Switch to that database
db.using_database("rag_db")

# ----- Create schema -----
fields = [
    FieldSchema("doc_id", DataType.INT64, is_primary=True, auto_id=False),
    FieldSchema("title", DataType.VARCHAR, max_length=200),
    FieldSchema("domain", DataType.VARCHAR, max_length=100),
    FieldSchema("content", DataType.VARCHAR, max_length=2000),
    FieldSchema("embedding", DataType.FLOAT_VECTOR, dim=384) 
]

schema = CollectionSchema(fields, description="Policy documents with embeddings")
collection = Collection("policy_docs_collection_2", schema)

# ----- Create index -----
index_params = {
    "index_type": "IVF_FLAT",
    "metric_type": "COSINE",
    "params": {"nlist": 128},
}
collection.create_index(field_name="embedding", index_params=index_params)


/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_13525/3144037655.py:8: PyMilvusDeprecationWarning: `db.using_database` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  db.using_database("rag_db")
/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_13525/3144037655.py:20: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection = Collection("policy_docs_collection_2", schema)
/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_13525/3144037655.py:28: PyMilvusDeprecationWarning: `Collection.create_index` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.create_index(field_name="embedding", index_params=index_params)


Status(code=0, message=)

In [9]:

# ----- Example data -----
content_chunks = [
    {
        "doc_id": 1,
        "section": "Pay Policies",
        "title": "Employee Pay Policy",
        "domain": "Human Resources",
        "content": "Employees are paid bi-weekly via direct deposit."
    },
    {
        "doc_id": 2,
        "section": "Leave of Absence",
        "title": "Leave Request and Approval Process",
        "domain": "Human Resources",
        "content": "Employees must submit a leave request for approval."
    },
    {
        "doc_id": 3,
        "section": "Internet Use",
        "title": "Acceptable Use of Company Internet",
        "domain": "IT & Security",
        "content": "Company internet must be used for work-related tasks only."
    },
    {
        "doc_id": 4,
        "section": "Break at Work",
        "title": "Employee Break Policy",
        "domain": "Workplace Operations",
        "content": "Employees can take an hour break."
    },
    {
        "doc_id": 5,
        "section": "Harassment",
        "title": "Workplace Harassment Policy",
        "domain": "Compliance",
        "content": "Interact with each employee with respect."
    }
]


# content_chunks_list = []
# for chunk in content_chunks:
#     content_chunks_list.append(chunk["content"])
content_chunks_list = [chunk["content"] for chunk in content_chunks]
print(content_chunks_list)


['Employees are paid bi-weekly via direct deposit.', 'Employees must submit a leave request for approval.', 'Company internet must be used for work-related tasks only.', 'Employees can take an hour break.', 'Interact with each employee with respect.']


In [10]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

doc_vectors = model.encode(content_chunks_list)
doc_vectors.shape

/Users/amiteshsinha/Training/ai_labs/2026_4_genAI_Lab/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0527 20:19:42.657707  866611 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(96, generation: 1)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15573.10it/s]


(5, 384)

In [11]:
# ---- Build columnar data ----
doc_ids = [int(i + 1) for i in range(len(content_chunks))]             # INT64
titles = [str(doc["title"]) for doc in content_chunks]                 # VARCHAR
domains = [str(doc["domain"]) for doc in content_chunks]               # VARCHAR
content = [str(doc["content"]) for doc in content_chunks]               # VARCHAR
embeddings = [list(map(float, vec)) for vec in doc_vectors]       # FLOAT_VECTOR(768)


# ---- Insert column-wise ----
collection.insert([doc_ids, titles, domains, content, embeddings])
# INSERT INTO policy_docs_collection (doc_id, title, domain, content, embedding) 
# VALUES (1, "Employee Pay Policy", "Human Resources", 
# "Employees are paid bi-weekly via direct deposit.", [0.1, 0.2, ..., 0.3])

collection.flush()

print(f"Successfully inserted {len(doc_ids)} documents into Milvus.")


/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_13525/643609608.py:10: PyMilvusDeprecationWarning: `Collection.insert` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.insert([doc_ids, titles, domains, content, embeddings])
/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_13525/643609608.py:15: PyMilvusDeprecationWarning: `Collection.flush` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.flush()


Successfully inserted 5 documents into Milvus.


In [12]:
#Load the collection before searching or querying
collection.load()
res = collection.query(expr="doc_id > 0", 
        output_fields=["doc_id", "title", "domain", "content", "embedding"])
print(res)


/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_13525/1248771852.py:2: PyMilvusDeprecationWarning: `Collection.load` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.load()
/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_13525/1248771852.py:3: PyMilvusDeprecationWarning: `Collection.query` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  res = collection.query(expr="doc_id > 0",


data: ["{'doc_id': 1, 'title': 'Employee Pay Policy', 'domain': 'Human Resources', 'content': 'Employees are paid bi-weekly via direct deposit.', 'embedding': [0.0247251708060503, -0.009081469848752022, 0.03887130320072174, 0.032301537692546844, -0.058905500918626785, 0.03912460058927536, 0.026017993688583374, -0.04369090870022774, -0.01916009932756424, -1.1305048701615306e-06, 0.01365350279957056, 0.004326740279793739, -0.05378558859229088, 0.006868922617286444, 0.007638321723788977, -0.060696400701999664, -0.037602175027132034, 0.007140923757106066, 0.13387171924114227, 0.0017353961011394858, -0.003075605956837535, -0.07781299203634262, -0.08451439440250397, -0.0032163704745471478, 0.1338280439376831, -0.03643588349223137, 0.016922777518630028, 0.04861478507518768, -0.04028793051838875, 0.019676748663187027, -0.043712858110666275, 0.051511410623788834, -0.01629961095750332, -0.03905921056866646, -0.004453877452760935, 0.0004745072219520807, -0.031205086037516594, 0.04895620048046112,

In [13]:
# print(utility.has_collection("demo_collection"))

# # Get details about a specific collection
# # Get collection details
# collection = Collection("demo_collection")  # instantiate the collection object
print(collection.schema)                    # show the schema
print(collection.num_entities)              # number of entities
print(collection.description)               # optional

{'auto_id': False, 'description': 'Policy documents with embeddings', 'fields': [{'name': 'doc_id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': False}, {'name': 'title', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 200}}, {'name': 'domain', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 100}}, {'name': 'content', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 2000}}, {'name': 'embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 384}}], 'enable_dynamic_field': False, 'enable_namespace': False}
5
Policy documents with embeddings


/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_13525/2025431850.py:7: PyMilvusDeprecationWarning: `Collection.num_entities` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  print(collection.num_entities)              # number of entities


## ONLINE PROCESSING

In [14]:
query = "What’s the leave policy?"
query_vector = model.encode([query])[0]
query_vector[:5]  # Show only first 5 values

array([0.0604582 , 0.03882854, 0.01172253, 0.0312459 , 0.1212422 ],
      dtype=float32)

In [ ]:
# Search for closest match only in the 'Human Resources' domain
results = collection.search(
    data=[query_vector],
    anns_field="embedding",
    param={"metric_type": "COSINE", "params": {"nprobe": 10}},
    limit=3,
    # expr='domain == "Human Resources"',
    output_fields=["doc_id", "title", "domain", "content"]
)

context_sring = ""
for res in results[0]:
    print(f"doc_id={res.entity.get('doc_id')}, "
          f"title={res.entity.get('title')}, "
          f"domain={res.entity.get('domain')}, "
          f"content={res.entity.get('content')}, "
          f"score={res.distance}")
    context_sring += f"\n -- \n {res.entity.get('content')} " # Append content to context string

print("\nContext String for RAG:\n", context_sring)    


doc_id=2, title=Leave Request and Approval Process, domain=Human Resources, content=Employees must submit a leave request for approval., score=0.6198837161064148
doc_id=1, title=Employee Pay Policy, domain=Human Resources, content=Employees are paid bi-weekly via direct deposit., score=0.23205110430717468

Context String for RAG:
 
 -- 
 Employees must submit a leave request for approval. 
 -- 
 Employees are paid bi-weekly via direct deposit. 


/var/folders/b8/45pl36ms0xx399j715gybz840000gn/T/ipykernel_13525/3539569942.py:2: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(


In [ ]:
from llm_utlity import ask_question_open_ai 

query = "What’s the leave policy?"
response = ask_question_open_ai(query, context_sring)
response


In [ ]:
print(f"User query: {query}")
print(f"Context: {context_sring}")

print(f"\n\nOpen AI Response: {response}")

In [ ]:
# RAG Evaluation
# Context_Recall: Did the system retrieve all relevant documents?
# Context_Precision: What proportion of retrieved documents are relevant?

input = "What’s the leave policy?"
expected_context = "Employees are entitled to 20 days of paid leave annually."
actual_context = '''
 Employees must submit a leave request for approval. 
 -- 
 Employees are paid bi-weekly via direct deposit. 
 '''
#Context_Recall: 0%


input = "What’s the leave policy?"
expected_context = ["Employees are entitled to 20 days of paid leave annually."," They must submit a leave request for approval."]
actual_context = ["Employees must submit a leave request for approval. ", " Employees are paid bi-weekly via direct deposit. ", " Company internet must be used for work-related tasks only. "]
#Context_Recall: 1 out of 2 expected = 1/2 = 50%


input = "What’s the leave policy?"
expected_context = ["Employees are entitled to 20 days of paid leave annually."," They must submit a leave request for approval."]
actual_context = ["They are entitled to 20 days of paid leave in a year."," They must submit a leave request for approval."]
#Context_Recall: 2 out of 2 expected = 2/2 = 100%

In [ ]:
# Context_Precision: What proportion of retrieved documents are relevant?
input = "What’s the leave policy?"
expected_context = ["Employees are entitled to 20 days of paid leave annually."," They must submit a leave request for approval."]
actual_context = ["Employees must submit a leave request for approval. ", " Employees are paid bi-weekly via direct deposit. ", " Company internet must be used for work-related tasks only. "]
#Context_Precision: 1 out of 3 retrieved = 1/3 = 33%


input = "What’s the leave policy?"
expected_context = ["Employees are entitled to 20 days of paid leave annually."," They must submit a leave request for approval."]
actual_context = ["They are entitled to 20 days of paid leave in a year."," They must submit a leave request for approval.", " Company internet must be used for work-related tasks only. "]
#Context_Precision: 2 out of 3 retrieved = 2/3 = 67%

In [ ]:
##### RAG Evaluation
### Retrieval Metrics
# Context_Recall: Did the system retrieve all relevant documents?
# Context_Precision: What proportion of retrieved documents are relevant?

### Generative Metrics
# Faithfulness - Does my answer accurately reflect the retrieved context? (Is the answer faithful to the retrieved information?)
# Accuracy - Does the answer accurately address the user's question based on the retrieved context? (Is the answer correct given the retrieved information?)


# F1 = 2 * (Context_Precision * Context_Recall) / (Context_Precision + Context_Recall) -- NOT USED

In [ ]:
### Generative Metrics
# Accuracy

input = "What’s the leave policy?"
context = ["Employees are entitled to 20 days of paid leave annually."," They must submit a leave request for approval."]

#expected_llm_output = '''
#llm as a judge
groundtruth = ''' 
    Based on the provided context, the leave policy states that employees are entitled to 20 days of paid leave annually 
    and must submit a leave request for approval.'''

actual_llm_output = '''
    Based on the provided context, the leave policy states that employees must submit a leave request for approval. 
    No other details are given. If you have more of the policy, I can summarize that as well.'''

accuracy = 0.50


# Faithfulness - Is the generated output consistent with the provided context?
faithfulness = 0.50